# Estimating a State-Space Model from Point Process Observations
> Smith, A.C., and Brown, E.N. (2003). Estimating a State-Space Model from Point Process Observations. Neural Computation 15, 965–991. https://doi.org/10.1162/089976603765202622.

## Latent model
$$x_k = \rho x_{k-1} + \alpha I_k + \epsilon_k $$

Variables:
- $x_k$ is the latent state at time $k\Delta$
- $\rho$ is the autoregressive parameter (correlation coefficient)
- $I_k$ is the input/stimulus indicator at time $k\Delta$ 
- $\alpha$ is the effect of the input on the latent state
- $\epsilon_k$ is the latent noise term (a Gaussian white noise with variance $\sigma_{\epsilon}^2$ and mean 0)

## Joint probability density
$$ p(x \mid \rho, \alpha, \sigma^2) = 
    \left[ \frac{(1 - \rho^2)}{2\pi\sigma_{\epsilon}^2} \right]^{\frac{1}{2}}
    \exp \left[ -0.5 \left( \frac{(1 - \rho^2)}{\sigma_{\epsilon}^2} x_0 + 
    \sum_{k=1}^K \frac{(x_k - \rho x_{k-1} - \alpha I_k)^2}{\sigma_{\epsilon}^2} \right)  \right]
$$

where $x = (x_0, x_1, \ldots, x_K)$ is the latent state vector.


## Derivation of the Recursive Nonlinear Filter Algorithm
Posterior prediction equation:
$$
p(x_k \mid H_k) = \frac{p(x_k \mid H_{k-1}) p(N(\Delta_k) \mid x_k, H_k)}{p(N(\Delta_k) \mid H_{k-1})}
$$
where the one-step prediction is:
$$
p(x_k \mid H_{k-1}) = \int p(x_k \mid x_{k-1}) p(x_{k-1} \mid H_{k-1}) dx_{k-1}
$$

Assume that at time $(k-1)\Delta$, we know $x_{k-1 \mid k-1}$ and $\sigma^2_{k-1 \mid k-1}$.

The transition density is
$p(x_k \mid x_{k-1 \mid k-1}) = \mathcal{N}(\rho x_{k-1 \mid k-1} + \alpha I_k, \sigma^2_{k \mid k-1})$ where $\sigma^2_{k \mid k-1} = \sigma^2_{\epsilon} + \rho^2 \sigma^2_{k-1 \mid k-1}$

The posterior probability density is:
$$
p(x_k \mid H_{k}) \propto 
    \exp \left[ -0.5 \frac{(x_k - \rho x_{k-1 \mid k-1} - \alpha I_k)^2}{\sigma^2_{k \mid k-1}} \right]
    \times
    \prod_{c=1}^{C} [\lambda_c(k \mid x_k) \Delta_k]^{N^c_k} \exp[-\lambda_c(k \mid x_k) \Delta_k]
$$

The log of the posterior probability density is:
$$
\log p(x_k \mid H_{k}) = 
    -0.5 \frac{(x_k - \rho x_{k-1 \mid k-1} - \alpha I_k)^2}{\sigma^2_{k \mid k-1}}
    + \sum_{c=1}^{C} N^c_k \log[\lambda_c(k \mid x_k) \Delta_k] - \lambda_c(k \mid x_k) \Delta_k
$$

We want to approximate the posterior probability density with a Gaussian distribution, so we differentiate the log of the posterior probability density with respect to $x_k$ and set it to zero:

$$
\frac{\partial}{\partial x_k} \log p(x_k \mid H_{k}) = 
    \frac{x_k - \rho x_{k-1 \mid k-1} - \alpha I_k}{\sigma^2_{k \mid k-1}}
    + \sum_{c=1}^{C} \frac{1}{\lambda_c(k \mid x_k)} \frac{\partial}{\partial x_k} (N^c_k - \lambda_c(k \mid x_k) \Delta_k) = 0
$$

Solving for $x_k$ we get:
$$

$$

In [60]:
import jax.numpy as jnp
import jax

n_cell_params = 2
n_cells = 2

rho = 0.0 # latent state correlation between time steps
alpha = 0.0  # effect of the input on the latent state
sigma_squared_epsilon = 1.0  # latent noise term

cell_params = jnp.zeros((n_cell_params, n_cells))

params = jnp.concatenate(
    (
        jnp.array([rho]),
        jnp.array([alpha]),
        jnp.array([sigma_squared_epsilon]),
        cell_params.flatten(),
    )
)

# unpack params
rho = params[0]
alpha = params[1]
sigma_squared_epsilon = params[2]
cell_params = params[3:].reshape(n_cell_params, n_cells)

design_matrix_t = jnp.ones((1, n_cell_params))
n_spikes_t = 1
input_t = 0
mode_prev = 0.0
sigma_squared_prev = sigma_squared_epsilon * 1 / ((1 - rho)**2)
dt = 0.002

conditional_intensity_t = jnp.exp(design_matrix_t @ cell_params).squeeze() * dt

# poisson_log_likelihood = jnp.sum(
#     jax.scipy.special.xlogy(n_spikes_t, conditional_intensity_t) - conditional_intensity_t
# )

poisson_log_likelihood = jnp.sum(
    jax.scipy.stats.poisson.logpmf(n_spikes_t, conditional_intensity_t)
)

one_step_mode = rho * mode_prev + alpha * input_t
one_step_variance = rho**2 * sigma_squared_prev + sigma_squared_epsilon

innovation = jnp.sum(cell_params @ (n_spikes_t - conditional_intensity_t))

posterior_mode = one_step_mode + one_step_variance * innovation
posterior_variance = -1 * 1 / (-1 / one_step_variance - jnp.sum(cell_params**2 @ conditional_intensity_t))
posterior_mode, posterior_variance

(Array(0., dtype=float32), Array(1., dtype=float32))